In [ ]:
import numpy as np

from pathlib import Path

import csv

import os

import re

import time

import random

from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import Select

In [3]:
cartella_sentenze = '/percorso/cartella_sentenze'

In [ ]:
log = "/percorso/log_sentenze_scaricate.csv"

sentenze = set()

with open(log, mode='r', newline='', encoding='utf-8') as file:
  lettore = csv.reader(file)
  for riga in lettore:
    sentenze.add(int(riga[0]))

print(len(sentenze))

In [5]:
sentenze_possibili = set(np.arange(1, 38185))

sentenze_disponibili = sentenze_possibili - sentenze

target_size = 3000

current_size = target_size - len(sentenze)

nuove_sentenze = set(np.random.choice(list(sentenze_disponibili), size=current_size, replace=False))

In [ ]:
opzioni = Options()
#opzioni.add_argument('--headless')
opzioni.add_argument('--no-sandbox')
opzioni.add_argument('--disable-dev-shm-usage')

preferenze = {'plugins.always_open_pdf_externally': True,
              'download.prompt_for_download': False,
              'download.default_directory': cartella_sentenze
              }
opzioni.add_experimental_option('prefs', preferenze)

driver = webdriver.Chrome(options=opzioni)
driver.get('https://www.italgiure.giustizia.it/sncass/')

scheda_principale = driver.current_window_handle

xpath_civile = "/html/body/div[1]/div[3]/form/table/tbody[2]/tr/td[1]/table/tbody/tr[2]/td/div/div/div/div/div/table/tbody/tr[1]/td[2]"

attesa1 = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, xpath_civile)))

civile = driver.find_element(By.XPATH, xpath_civile)
civile.click()

time.sleep(0.5)

for riga in nuove_sentenze:

  try:

    searchbar = driver.find_element(By.ID, 'searchterm')
    searchbar.clear()
    searchbar.send_keys(f"{riga}/2022")
    searchbar.send_keys(Keys.RETURN)

    time.sleep(2.0)

    #menu_ordinamento = driver.find_element(By.XPATH, "//option[@id='sort_datdep']/parent::select")
    #selettore = Select(menu_ordinamento)
    #selettore.select_by_value("pd desc,numdec desc")

    attesa2 = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, 'text2pdf')))

    check = driver.find_element(By.XPATH, '/html/body/div[1]/div[3]/form/table/tbody[2]/tr/td[2]/div/div[2]/div[1]/div/h3/a/span[3]')

    if check.text in ["Ordinanza Interlocutoria", "Decreto"]:
      continue

    try:
    
      oscuramento = driver.find_element(By.CLASS_NAME, 'text2ocr')

      if 'oscuramento' in oscuramento.text:
        continue
  
    except:
      pass

    documento = driver.find_element(By.CLASS_NAME, 'text2pdf')
    documento.click()

    time.sleep(0.5)

    lista_pdf = os.listdir('/Users/maxiorsini/Desktop/progettocooking/progettonorm/sentenze')

    while any(re.search(r'\.crdownload$', s) for s in lista_pdf) == True:
      lista_pdf = os.listdir('/Users/maxiorsini/Desktop/progettocooking/progettonorm/sentenze')
      time.sleep(0.01)

    schede = driver.window_handles

    for scheda in schede:
      if scheda != scheda_principale:
        driver.switch_to.window(scheda)
        driver.close()
        driver.switch_to.window(scheda_principale)

    with open(log, mode='a', newline='', encoding='utf-8') as file:
      scrittore = csv.writer(file)
      scrittore.writerow([riga])

    time.sleep(random.uniform(3.0,5.0))
  
  except:

    azzera_filtri = driver.find_element(By.XPATH, "/html/body/div[1]/div[3]/form/table/tbody[1]/tr[2]/td[1]/button")
    azzera_filtri.click()
    xpath_civile = "/html/body/div[1]/div[3]/form/table/tbody[2]/tr/td[1]/table/tbody/tr[2]/td/div/div/div/div/div/table/tbody/tr[1]/td[2]"

    attesa1 = WebDriverWait(driver, 20).until(EC.element_to_be_clickable((By.XPATH, xpath_civile)))

    civile = driver.find_element(By.XPATH, xpath_civile)
    civile.click()

  #26315/2022
